In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
from tqdm import tqdm
import requests
import re

# ------------------------
# 모델 로드
# ------------------------
model_name = "MLP-KTLim/llama-3-Korean-Bllossom-8B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/710 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

In [2]:
DEV_URL  = "https://raw.githubusercontent.com/beefed-up-geek/HCLT-KACL-2025/main/Korean_Inappropriate_Detection/dataset/original_formatted/dev.json"
TEST_URL = "https://raw.githubusercontent.com/beefed-up-geek/HCLT-KACL-2025/main/Korean_Inappropriate_Detection/dataset/original_formatted/test.json"

def load_json_from_url(url: str):
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.json()

dev_data  = load_json_from_url(DEV_URL)
test_data = load_json_from_url(TEST_URL)

print(f"Dev samples:  {len(dev_data)}")
print(f"Test samples: {len(test_data)}")


Dev samples:  101
Test samples: 397


In [19]:
SYSTEM_PROMPT = """대화 맥락을 읽고, 제시된 '판별 대상 발화' 하나가 부적절한지 판별하라.
1 = 부적절(inappropriate), 0 = 적절(appropriate).
정답은 한 자리 숫자(0 또는 1)만 출력하라.

Read the dialogue context, and judge if the target utterance is inappropriate.
1 = inappropriate, 0 = appropriate.
Answer with only one digit: 0 or 1."""

def split_dialogue_lines(dialogue: str):
    """줄바꿈 기준으로 화자별 발화를 분리. 공백 줄 제거."""
    return [ln for ln in dialogue.split("\n") if ln.strip() != ""]

def build_user_prompt(full_dialogue_text: str, target_utt_text: str) -> str:
    """
    ID는 절대 포함하지 않는다. (문맥+대상발화 텍스트만)
    [전체 대화] ...  [판별 대상 발화] ...
    """
    return (
        "다음은 하나의 대화 전체 내용이다.\n"
        "[전체 대화]\n"
        f"{full_dialogue_text}\n\n"
        "[판별 대상 발화]\n"
        f"{target_utt_text}\n\n"
        "정답: (0 또는 1 중 하나의 숫자만)"
    )

def parse_single_digit_from_generated_text(gen_text: str) -> int:
    """
    새로 생성된 텍스트에서 첫 0/1 한 자리만 채택. 없으면 0.
    """
    m = re.search(r"[01]", gen_text)
    return int(m.group(0)) if m else 0

@torch.no_grad()
def predict_single_utterance(full_dialogue_text: str, target_utt_text: str) -> int:
    """
    한 발화에 대해 LLM 1회 호출 → 0/1 정수 반환.
    LLM의 '새로 생성된 토큰'만 디코딩하여 파싱.
    """
    user_prompt = build_user_prompt(full_dialogue_text, target_utt_text)
    prompt = f"{SYSTEM_PROMPT}\n\n{user_prompt}"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=3,        # 한 자리면 충분
        temperature=0.0,         # 결정적
        top_p=1.0,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # === 생성된 토큰만 추출 ===
    prompt_len = inputs["input_ids"].shape[-1]
    new_tokens = outputs[0][prompt_len:]
    gen_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return parse_single_digit_from_generated_text(gen_text)


In [21]:
LABEL_TO_INT = {"appropriate": 0, "inappropriate": 1}
INT_TO_LABEL = {0: "appropriate", 1: "inappropriate"}

def make_utt_id(dialogue_id: str, turn_idx: int) -> str:
    return f"{dialogue_id}-{turn_idx:03d}"

y_true, y_pred = [], []

for item in tqdm(dev_data, desc="Evaluate dev"):
    dialogue_id = item["id"]
    full_dialogue_text = item["dialogue"]
    lines = split_dialogue_lines(full_dialogue_text)

    # gold 라벨 맵 (평가에만 사용; 프롬프트에는 쓰지 않음)
    gold_map = {o["id"]: LABEL_TO_INT[o["label"]] for o in item["output"]}

    for i, line in enumerate(lines, start=1):
        utt_id = make_utt_id(dialogue_id, i)
        if utt_id not in gold_map:
            continue

        # "화자1: 내용" → 내용만 추출 (ID 금지)
        target_utt_text = line.split(": ", 1)[-1] if ": " in line else line

        pred = predict_single_utterance(full_dialogue_text, target_utt_text)
        y_pred.append(pred)
        y_true.append(gold_map[utt_id])

acc = sum(int(p == t) for p, t in zip(y_pred, y_true)) / max(1, len(y_true))
print(f"Dev Accuracy: {acc:.4f}  ({sum(int(p == t) for p,t in zip(y_pred,y_true))}/{len(y_true)})")


Evaluate dev:   0%|          | 0/101 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Evaluate dev:   1%|          | 1/101 [00:00<00:24,  4.09it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Evaluate dev:   2%|▏         | 2/101 [00:00<00:37,  2.67it/s]The following generat

Dev Accuracy: 0.6210  (254/409)


In [22]:
submission = []

for item in tqdm(test_data, desc="Infer test"):
    dialogue_id = item["id"]
    full_dialogue_text = item["dialogue"]
    lines = split_dialogue_lines(full_dialogue_text)

    outputs = []
    for i, line in enumerate(lines, start=1):
        utt_id = make_utt_id(dialogue_id, i)  # 제출 포맷용
        target_utt_text = line.split(": ", 1)[-1] if ": " in line else line

        pred_int = predict_single_utterance(full_dialogue_text, target_utt_text)
        pred_label = INT_TO_LABEL[pred_int]  # 필요 시 정수로 바꿔도 됨

        outputs.append({
            "id": utt_id,
            "label": pred_label   # 혹시 대회가 0/1 정수 요구면 pred_int로 교체
        })

    submission.append({
        "id": dialogue_id,
        "output": outputs
    })

SUBMIT_PATH = "submission.json"
with open(SUBMIT_PATH, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print("Saved:", SUBMIT_PATH)


Infer test:   0%|          | 0/397 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Infer test:   0%|          | 1/397 [00:00<02:59,  2.20it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set 

Saved: submission.json
